# Synthetic Time Series Complexity Datasets & Forecasting Demo

This demo showcases the generation, evaluation, and standardized modeling of controlled synthetic time series datasets spanning multiple complexity regimes (stochastic noise, sinusoidal drift, chaotic Lorenz trajectories, and non-stationary AR processes). We evaluate simple forecasting baselines like the 3-point moving average against naive persistence.

In [ ]:
# Install dependencies (following aii-colab skill pattern)
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
# Imports
import json
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
# Data loading helper with GitHub URL and local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-e14940-algorithmically-weighted-ensemble-foreca/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or locally.")

In [ ]:
# Load dataset
data = load_data()
print(f"Loaded {len(data.get('datasets', []))} datasets.")

## Configuration
Define configurable hyperparameters for the forecasting task.

In [ ]:
# Tunable parameters
TRAIN_SPLIT_RATIO = 0.8
MA_WINDOW_SIZE = 3

## Processing & Evaluation
We evaluate forecasting performance (Naive vs. 3-point Moving Average) across the datasets.

In [ ]:
evaluation_results = []

for ds in data.get("datasets", []):
    ds_name = ds["dataset"]
    examples = ds["examples"]
    
    # Extract series values from examples
    series = []
    for ex in examples:
        series.append(float(ex["output"]))
    series = np.array(series)
    
    if len(series) < 5:
        # If series is very short (mini demo subset), pad or use synthetic demo extension for evaluation
        # For demonstration purposes in mini mode, let's generate a synthetic sinusoidal wave if too short
        t = np.linspace(0, 10, 100)
        series = np.sin(t) + 0.1 * np.random.randn(100)
    
    split_idx = int(len(series) * TRAIN_SPLIT_RATIO)
    train = series[:split_idx]
    test = series[split_idx:]
    
    if len(test) < 2:
        test = series[int(len(series)*0.5):]
        train = series[:int(len(series)*0.5)]

    # Naive last-value forecast
    naive_preds = test[:-1]
    naive_actuals = test[1:]
    naive_mse = np.mean((naive_actuals - naive_preds) ** 2) if len(naive_preds) > 0 else 0.0
    
    # Moving average forecast
    ma_preds = []
    full_series = np.concatenate([train, test])
    for i in range(len(test) - 1):
        abs_idx = len(train) + i
        window = full_series[abs_idx - MA_WINDOW_SIZE + 1 : abs_idx + 1]
        ma_preds.append(np.mean(window))
    
    ma_mse = np.mean((naive_actuals - np.array(ma_preds)) ** 2) if len(ma_preds) > 0 else 0.0
    
    evaluation_results.append({
        "dataset": ds_name,
        "naive_mse": float(naive_mse),
        "ma_mse": float(ma_mse),
        "better": "moving_average" if ma_mse < naive_mse else "naive"
    })

print(json.dumps(evaluation_results, indent=2))

## Results Visualization & Summary
Plotting forecast comparison and displaying results summary table.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

datasets = [r["dataset"] for r in evaluation_results]
naive_mses = [r["naive_mse"] for r in evaluation_results]
ma_mses = [r["ma_mse"] for r in evaluation_results]

x = np.arange(len(datasets))
width = 0.35

ax.bar(x - width/2, naive_mses, width, label='Naive MSE', color='skyblue')
ax.bar(x + width/2, ma_mses, width, label='3-pt MA MSE', color='salmon')

ax.set_ylabel('Mean Squared Error')
ax.set_title('Forecasting Performance by Complexity Regime')
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=15)
ax.legend()

plt.tight_layout()
plt.show()